# 03 — Métricas Derivadas

**Objetivo:** calcular indicadores que tornam os dados comparáveis e interpretáveis.
**Entrada:** DataFrame de `limpar_dados()` — formato long, tipos corretos.
**Saída:** mesmo DataFrame com três novas colunas: `taxa_100k`, `media_movel_3`, `variacao_pct`.
**Próximo passo:** copiar o consolidado para `gerar_metricas()` em `pipeline.py`.

## Célula 1 — Setup

Partimos do dado já limpo — `gerar_metricas()` recebe a saída de `limpar_dados()`.
Importar do `pipeline` evita duplicar código e garante que o notebook teste
exatamente a função que entrará em produção.

In [2]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.append(str(RAIZ))

from pipeline import carregar_dados, limpar_dados

ARQUIVO = RAIZ / "data" / "raw" / "BaseDPEvolucaoMensalCisp.csv"

df_limpo = limpar_dados(carregar_dados(ARQUIVO))
print("Entrada:", df_limpo.shape)
df_limpo.head(5)

2026-05-11 19:11:49 [INFO] Dados carregados: 37588 linhas, 65 colunas
2026-05-11 19:11:49 [INFO] Limpeza concluida. Shape final: (150352, 8)


Entrada: (150352, 8)


,cisp,mes_ano,aisp,risp,munic,regiao,tipo_crime,qtd_ocorrencias
0,1,2003-01,5,1,RIO DE JANEIRO,CAPITAL,hom_doloso,0
1,4,2003-01,5,1,RIO DE JANEIRO,CAPITAL,hom_doloso,3
2,5,2003-01,5,1,RIO DE JANEIRO,CAPITAL,hom_doloso,3
3,6,2003-01,1,1,RIO DE JANEIRO,CAPITAL,hom_doloso,6
4,7,2003-01,1,1,RIO DE JANEIRO,CAPITAL,hom_doloso,4


## Célula 2 — Inspecionar o dado de entrada

Antes de criar novas colunas, confirmamos o formato do DataFrame que chegou:
quais AISPs existem, qual o range de datas, e como ficou o formato long.

Essa inspeção determina *como* calcular a taxa por 100k:
precisamos saber exatamente quais valores de AISP existem para mapear
as populações corretamente.

In [3]:
print("AISPs disponíveis:", sorted(df_limpo["aisp"].unique()))
print()
print("Período:", df_limpo["mes_ano"].min(), "→", df_limpo["mes_ano"].max())
print()
print("Crimes:", df_limpo["tipo_crime"].unique())

AISPs disponíveis: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '43', '5', '6', '7', '8', '9']

Período: 2003-01 → 2026-03

Crimes: <StringArray>
['hom_doloso', 'latrocinio', 'cvli', 'letalidade_violenta']
Length: 4, dtype: str


## Célula 3 — Taxa por 100k habitantes

**Por que normalizar?**
Comparar ocorrências brutas entre AISPs é enganoso: a AISP com mais
crimes pode simplesmente ter mais habitantes. A taxa por 100 mil habitantes
remove esse viés de tamanho — é o indicador padrão em segurança pública.

**Fórmula:**
```
taxa_100k = (qtd_ocorrencias / populacao_aisp) × 100.000
```

Usamos um dicionário `{aisp: populacao}` com estimativas do IBGE 2022.
AISPs não cadastradas recebem `NaN` — melhor expor a lacuna do que inventar
um número. O `np.where()` ou `.fillna()` podem tratar isso se necessário.

In [4]:
df = df_limpo.copy()

# Estimativas populacionais por AISP — fonte: IBGE 2022 (aproximado)
POPULACAO_AISP = {
    "1":  232000,  "2":  340000,  "3":  250000,  "4":  214000,
    "5":  530000,  "6":  371000,  "7":  275000,  "9":  226000,
    "10": 195000,  "14": 417000,  "15": 364000,  "16": 389000,
    "17": 295000,  "18": 231000,  "19": 295000,  "20": 305000,
    "21": 248000,  "22": 377000,  "23": 264000,  "24": 237000,
    "25": 229000,  "26": 188000,  "27": 232000,  "28": 278000,
    "29": 316000,  "30": 275000,  "31": 196000,  "32": 228000,
    "33": 253000,  "34": 213000,  "35": 196000,  "36": 228000,
    "37": 180000,  "38": 243000,  "39": 302000,  "40": 198000,
    "41": 310000,  "43": 216000,
}

df["populacao"] = df["aisp"].map(POPULACAO_AISP)

sem_pop = df[df["populacao"].isna()]["aisp"].unique()
if len(sem_pop):
    print("AISPs sem população cadastrada:", sem_pop)
else:
    print("Todas as AISPs têm população cadastrada.")

df["taxa_100k"] = (df["qtd_ocorrencias"] / df["populacao"]) * 100_000

print()
print("taxa_100k — amostra:")
df[["aisp", "mes_ano", "tipo_crime", "qtd_ocorrencias", "populacao", "taxa_100k"]].head(6)

AISPs sem população cadastrada: <StringArray>
['12', '11', '8', '13']
Length: 4, dtype: str

taxa_100k — amostra:


,aisp,mes_ano,tipo_crime,qtd_ocorrencias,populacao,taxa_100k
0,5,2003-01,hom_doloso,0,530000.0,0.000000
1,5,2003-01,hom_doloso,3,530000.0,0.566038
2,5,2003-01,hom_doloso,3,530000.0,0.566038
3,1,2003-01,hom_doloso,6,232000.0,2.586207
4,1,2003-01,hom_doloso,4,232000.0,1.724138
5,2,2003-01,hom_doloso,1,340000.0,0.294118


## Célula 4 — Média móvel de 3 meses

Uma série mensal de crimes é ruidosa — um mês atípico pode distorcer a leitura.
A média móvel suaviza esse ruído calculando a média dos últimos N meses.

**Cuidados importantes:**
- **Ordenar primeiro** por `["aisp", "tipo_crime", "mes_ano"]` — sem isso a janela
  deslizante pode pegar meses fora de ordem e o resultado será incorreto.
- **`groupby()` antes do `.transform()`** — garante que a janela não "vaze" entre
  AISPs diferentes. Sem groupby, o último mês da AISP 1 entraria na média da AISP 2.
- **`min_periods=1`** — evita que os dois primeiros meses de cada grupo virem NaN;
  com menos de 3 meses disponíveis, usa o que tem.

In [5]:
df.sort_values(["aisp", "tipo_crime", "mes_ano"], inplace=True)

df["media_movel_3"] = (
    df.groupby(["aisp", "tipo_crime"])["qtd_ocorrencias"]
      .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

# Verificação: AISP 1, hom_doloso — comparar qtd com média móvel
amostra = df[(df["aisp"] == "1") & (df["tipo_crime"] == "hom_doloso")].head(6)
print(amostra[["mes_ano", "qtd_ocorrencias", "media_movel_3"]].to_string(index=False))

mes_ano  qtd_ocorrencias  media_movel_3
2003-01                6       6.000000
2003-01                4       5.000000
2003-02                6       5.333333
2003-02                1       3.666667
2003-03                3       3.333333
2003-03                0       1.333333


## Célula 5 — Variação percentual mês a mês

`pct_change()` calcula `(valor_atual − valor_anterior) / valor_anterior × 100`.
Aplicado via `groupby`, garante que a variação não compare o último mês de
uma AISP com o primeiro mês da próxima.

**O primeiro registro de cada grupo sempre será `NaN`** — não há mês anterior
para comparar. Isso é esperado e correto; não preencha com 0, pois zero
significaria "sem variação", o que seria falso.

In [6]:
df["variacao_pct"] = (
    df.groupby(["aisp", "tipo_crime"])["qtd_ocorrencias"]
      .pct_change() * 100
)

# Verificação: primeiro mês deve ser NaN, demais com percentuais
amostra = df[(df["aisp"] == "1") & (df["tipo_crime"] == "hom_doloso")].head(6)
print(amostra[["mes_ano", "qtd_ocorrencias", "variacao_pct"]].to_string(index=False))

mes_ano  qtd_ocorrencias  variacao_pct
2003-01                6           NaN
2003-01                4    -33.333333
2003-02                6     50.000000
2003-02                1    -83.333333
2003-03                3    200.000000
2003-03                0   -100.000000


## Célula 6 — Validação final

Antes de passar o DataFrame adiante, conferimos se as três métricas fazem sentido:
- `taxa_100k`: valores razoáveis (crimes raros têm taxa baixa)
- `media_movel_3`: sempre ≥ 0 e suaviza picos
- `variacao_pct`: NaN apenas no primeiro mês de cada grupo, percentuais no resto

In [7]:
print("Estatísticas descritivas das métricas:")
print(df[["taxa_100k", "media_movel_3", "variacao_pct"]].describe().round(2))
print()
print("Nulos por coluna nova:")
print(df[["taxa_100k", "media_movel_3", "variacao_pct"]].isnull().sum())
print()
print("Shape final:", df.shape)

Estatísticas descritivas das métricas:
       taxa_100k  media_movel_3  variacao_pct
count  129404.00      150352.00      98097.00
mean        0.97           2.43           inf
std         1.65           3.51           NaN
min         0.00           0.00       -100.00
25%         0.00           0.00        -83.33
50%         0.32           1.00          0.00
75%         1.29           3.33        450.00
max        31.36          42.67           inf

Nulos por coluna nova:
taxa_100k        20948
media_movel_3        0
variacao_pct     52255
dtype: int64

Shape final: (150352, 12)


c:\Users\arthu\OneDrive\Desktop\Projetos\isp-analise\.venv\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


---
## Consolidado — o que vai para `gerar_metricas()` no pipeline.py

```python
POPULACAO_AISP = {
    "1":  232000,  "2":  340000,  "3":  250000,  "4":  214000,
    "5":  530000,  "6":  371000,  "7":  275000,  "9":  226000,
    "10": 195000,  "14": 417000,  "15": 364000,  "16": 389000,
    "17": 295000,  "18": 231000,  "19": 295000,  "20": 305000,
    "21": 248000,  "22": 377000,  "23": 264000,  "24": 237000,
    "25": 229000,  "26": 188000,  "27": 232000,  "28": 278000,
    "29": 316000,  "30": 275000,  "31": 196000,  "32": 228000,
    "33": 253000,  "34": 213000,  "35": 196000,  "36": 228000,
    "37": 180000,  "38": 243000,  "39": 302000,  "40": 198000,
    "41": 310000,  "43": 216000,
}

def gerar_metricas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["populacao"] = df["aisp"].map(POPULACAO_AISP)
    df["taxa_100k"] = (df["qtd_ocorrencias"] / df["populacao"]) * 100_000

    df.sort_values(["aisp", "tipo_crime", "mes_ano"], inplace=True)

    df["media_movel_3"] = (
        df.groupby(["aisp", "tipo_crime"])["qtd_ocorrencias"]
          .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    )

    df["variacao_pct"] = (
        df.groupby(["aisp", "tipo_crime"])["qtd_ocorrencias"]
          .pct_change() * 100
    )

    log.info("Métricas geradas. Shape final: %s", df.shape)
    return df
```